# ErgoVision — Ergonomic Risk Assessment Pipeline

Vision-based ergonomic risk assessment using YOLOv8-pose + RULA-inspired scoring.

**Pipeline**:  Image → YOLOv8-pose → keypoints → ergonomic angles → risk classification

---

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless numpy

## 2. Locate the dataset

The code inspects the directory automatically — no hardcoded folder structure.

In [ ]:
import os
from pathlib import Path

# Possible Kaggle mount points
candidates = [
    Path('/kaggle/input'),
    Path('../input'),
    Path('.'),
]

dataset_path = None
for p in candidates:
    if p.exists():
        # Walk a level deep to find actual image files
        for root, dirs, files in os.walk(p):
            if any(f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp')) for f in files):
                dataset_path = Path(root)
                break
        if dataset_path:
            break

if dataset_path is None:
    raise FileNotFoundError(
        "Could not locate dataset. Ensure the Kaggle dataset "
        "'melsmm/posture-keypoints-detection' is added to the notebook."
    )

print(f'Dataset found at: {dataset_path}')

## 3. Install ErgoVision package

Mount the package from the current directory (works on Kaggle if uploaded as a dataset, or locally).

In [ ]:
# The ergovision/ folder is expected in the current working directory.
# On Kaggle you can upload it as a dataset and symlink or copy it, or use
# a self-contained import by adding the path to sys.path.
import sys
sys.path.insert(0, '.')

# Verify the package is accessible
import ergovision
print(f'ErgoVision v{ergovision.__version__} loaded')
print(f'Package location: {ergovision.__file__}')

## 4. Run the pipeline

Process a subset of 50 images.  The pipeline will:
- find images in the dataset
- run YOLOv8-pose inference
- compute ergonomic risk scores
- save annotated visuals, CSV, and JSON

In [ ]:
from ergovision.pipeline import ErgoPipeline

pipeline = ErgoPipeline()
results = pipeline.run(
    dataset_path=dataset_path,
    subset_size=50,
    save_visualizations=True,
    verbose=True,
)

## 5. Preview results

Display a few annotated images and the CSV summary.

In [ ]:
import csv
import matplotlib.pyplot as plt
from pathlib import Path
import cv2

# Show CSV contents
csv_path = Path('outputs') / 'ergonomic_assessment.csv'
if csv_path.exists():
    with open(csv_path) as f:
        reader = csv.DictReader(f)
        rows = list(reader)
    print(f'CSV rows: {len(rows)}')
    if rows:
        print(f'{"image":35s} {"risk_class":15s} {"score":8s} {"torso":8s} {"neck":8s}')
        print('-' * 74)
        for r in rows[:10]:
            name = Path(r['image_path']).name[:34]
            print(f'{name:35s} {r["risk_class"]:15s} {r["risk_score"]:8s} '
                  f'{r["torso_angle_value"]:8s} {r["neck_angle_value"]:8s}')
        if len(rows) > 10:
            print(f'... and {len(rows) - 10} more rows')

# Show a few visualisations
viz_dir = Path('outputs') / 'visualizations'
viz_files = sorted(viz_dir.glob('*'))[:6]

if viz_files:
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, vf in zip(axes.flat, viz_files):
        img = cv2.cvtColor(cv2.imread(str(vf)), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(vf.name, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No visualisations saved yet.')

## 6. Risk distribution chart

In [ ]:
from collections import Counter

classes = [r['risk_class'] for r in rows]
counts = Counter(classes)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#2ecc71', '#f1c40f', '#e74c3c']
bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='gray')
ax.set_ylabel('People detected')
ax.set_title('Ergonomic Risk Distribution')
for bar, cnt in zip(bars, counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(cnt), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

## Summary

All outputs have been saved to the `outputs/` directory:
- `visualizations/` — annotated images with skeleton + risk overlay
- `ergonomic_assessment.csv` — per-person scores and feature values
- `ergonomic_assessment.json` — full structured results